# One-Step Fall Detection — Model Testing
**COS30018 – Intelligent Systems | Assignment Topic 3**

This notebook tests the already-trained YOLOv8 one-step fall detection model.
No training is done here — just load the best weights and evaluate.

| Class ID | Label |
|----------|-------|
| 0 | fall detected |
| 1 | walk |
| 2 | sit |

---
## 1. Imports

In [1]:
import random
from pathlib import Path
from collections import Counter

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

from ultralytics import YOLO

random.seed(42)
print('Imports OK')

Imports OK


In [3]:
import glob
results = glob.glob('runs/**/*.pt', recursive=True)
for r in results:
    print(r)

runs\detect\runs\one_step\fall_detection\weights\best.pt
runs\detect\runs\one_step\fall_detection\weights\epoch0.pt
runs\detect\runs\one_step\fall_detection\weights\epoch10.pt
runs\detect\runs\one_step\fall_detection\weights\epoch20.pt
runs\detect\runs\one_step\fall_detection\weights\epoch30.pt
runs\detect\runs\one_step\fall_detection\weights\epoch40.pt
runs\detect\runs\one_step\fall_detection\weights\epoch50.pt
runs\detect\runs\one_step\fall_detection\weights\epoch60.pt
runs\detect\runs\one_step\fall_detection\weights\epoch70.pt
runs\detect\runs\one_step\fall_detection\weights\epoch80.pt
runs\detect\runs\one_step\fall_detection\weights\epoch90.pt
runs\detect\runs\one_step\fall_detection\weights\last.pt


---
## 2. Configuration — Edit Paths Here

In [5]:
# ── Path to your trained best.pt ─────────────────────────────────────────────
BEST_WEIGHTS = Path(r'runs\detect\runs\one_step\fall_detection\weights\best.pt')

# ── Dataset paths ─────────────────────────────────────────────────────────────
DATASET_ROOT = Path('dataset/fall_dataset')
TEST_IMG_DIR = DATASET_ROOT / 'images' / 'test'
TEST_LBL_DIR = DATASET_ROOT / 'labels' / 'test'
DATA_YAML    = DATASET_ROOT / 'data.yaml'

# ── Model settings ────────────────────────────────────────────────────────────
CLASS_NAMES       = ['fall detected', 'walk', 'sit']
IMG_SIZE          = 640
CONF_THRESHOLD    = 0.25    # minimum confidence to count as detection
IOU_THRESHOLD     = 0.45

# ── Low-light filename prefixes ───────────────────────────────────────────────
LOW_LIGHT_PREFIXES = ('fall_night', 'low_light_walk', 'sit_lowlight')

# ── Box colours (RGB) ─────────────────────────────────────────────────────────
BOX_COLORS = {
    'fall detected': (231, 76,  60),
    'walk':          (52,  152, 219),
    'sit':           (46,  204, 113),
}

# ── Verify weights exist ──────────────────────────────────────────────────────
assert BEST_WEIGHTS.exists(), f'Weights not found: {BEST_WEIGHTS}'
print(f'Model weights : {BEST_WEIGHTS}')
print(f'Test images   : {TEST_IMG_DIR}')
print(f'Data yaml     : {DATA_YAML}')

Model weights : runs\detect\runs\one_step\fall_detection\weights\best.pt
Test images   : dataset\fall_dataset\images\test
Data yaml     : dataset\fall_dataset\data.yaml


---
## 3. Load Trained Model

In [6]:
model = YOLO(str(BEST_WEIGHTS))

# Print model summary
print(f'Model loaded  : {BEST_WEIGHTS.name}')
print(f'Classes       : {model.names}')
print(f'Parameters    : {sum(p.numel() for p in model.model.parameters()):,}')

Model loaded  : best.pt
Classes       : {0: 'fall detected', 1: 'walk', 2: 'sit'}
Parameters    : 11,136,761


---
## 4. Overall Test Set Evaluation
Runs the official YOLO validation loop on the test split — gives mAP, precision, recall per class.

In [7]:
metrics = model.val(
    data    = str(DATA_YAML),
    split   = 'test',
    imgsz   = IMG_SIZE,
    conf    = CONF_THRESHOLD,
    iou     = IOU_THRESHOLD,
    plots   = True,
    verbose = True,
)

print('\n' + '='*45)
print('       OVERALL TEST SET METRICS')
print('='*45)
print(f'  mAP@50       : {metrics.box.map50:.4f}')
print(f'  mAP@50-95    : {metrics.box.map:.4f}')
print(f'  Precision    : {metrics.box.mp:.4f}')
print(f'  Recall       : {metrics.box.mr:.4f}')
print('='*45)

print('\n  PER-CLASS AP@50')
print('-'*35)
for name, ap in zip(CLASS_NAMES, metrics.box.ap50):
    bar = '█' * int(ap * 30)
    print(f'  {name:<16} {ap:.4f}  {bar}')

Ultralytics 8.4.33  Python-3.10.11 torch-2.11.0+cpu CPU (Intel Core Ultra 5 125H)
Model summary (fused): 73 layers, 11,126,745 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access  (ping: 0.50.1 ms, read: 99.3199.4 MB/s, size: 1257.8 KB)
val: Scanning C:\Users\celes\Downloads\COS30018 Intelligent System\Assignment\dataset\fall_dataset\labels\test.cache... 183 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 183/183  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 12/12 2.0s/it 24.3s2.2s
                   all        183        223      0.442      0.598       0.48       0.26
         fall detected         61         64      0.565      0.547      0.561       0.28
                  walk         69         89       0.36      0.775       0.47      0.275
                   sit         65         70      0.402      0.471       0.41      0.226
Speed: 0.7ms preprocess, 114.4ms inference, 0.0ms loss, 0.8ms postprocess per

In [12]:
import numpy as np
import matplotlib.pyplot as plt

# ── What we actually have ─────────────────────────────────────────────────────
ap50    = metrics.box.ap50   # AP at IoU=0.50  per class  → (3,)
ap5095  = metrics.box.ap     # mean AP 0.50-0.95 per class → (3,)

colors = ['#e74c3c', '#3498db', '#2ecc71']

# ── Table ─────────────────────────────────────────────────────────────────────
print('='*55)
print('   IoU & mAP Results — One-Step YOLOv8')
print('='*55)
print(f"  {'Class':<16} {'AP@50':>8} {'AP@50-95':>10}")
print('-'*55)
for name, a50, a5095 in zip(CLASS_NAMES, ap50, ap5095):
    print(f"  {name:<16} {a50:>8.4f} {a5095:>10.4f}")
print('-'*55)
print(f"  {'mAP (mean)':<16} {metrics.box.map50:>8.4f} {metrics.box.map:>10.4f}")
print('='*55)
print(f"\n  Interpretation:")
print(f"  AP@50    = detection counted correct if IoU ≥ 0.50")
print(f"  AP@50-95 = stricter — averaged over IoU 0.50 to 0.95")
print(f"  Drop from AP@50 → AP@50-95 shows bounding box precision")

# ── Plot 1: AP@50 vs AP@50-95 per class ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
x = np.arange(len(CLASS_NAMES))
w = 0.35
b1 = ax.bar(x - w/2, ap50,   w, label='AP@50',    color=colors, edgecolor='black', alpha=0.9)
b2 = ax.bar(x + w/2, ap5095, w, label='AP@50-95', color=colors, edgecolor='black', alpha=0.45)
ax.bar_label(b1, fmt='%.4f', padding=3, fontsize=9)
ax.bar_label(b2, fmt='%.4f', padding=3, fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES, fontsize=10)
ax.set_ylabel('AP', fontsize=11)
ax.set_title('AP@50 vs AP@50-95 — Per Class', fontsize=12)
ax.set_ylim(0, 1.0)
ax.axhline(y=metrics.box.map50, color='black', linestyle='--',
           linewidth=1.2, label=f'mAP@50 = {metrics.box.map50:.4f}')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

# ── Plot 2: Drop from AP@50 to AP@50-95 (shows localisation difficulty) ──────
ax2 = axes[1]
drop = ap50 - ap5095
bars = ax2.bar(CLASS_NAMES, drop, color=colors, edgecolor='black', alpha=0.85)
ax2.bar_label(bars, fmt='%.4f', padding=3, fontsize=9)
ax2.set_ylabel('AP Drop (AP@50 − AP@50-95)', fontsize=11)
ax2.set_title('Localisation Difficulty per Class\n(larger drop = harder to localise precisely)', fontsize=11)
ax2.set_ylim(0, max(drop) * 1.3)
ax2.grid(axis='y', alpha=0.3)

# Annotate percentage drop
for i, (d, a) in enumerate(zip(drop, ap50)):
    pct = d / a * 100 if a > 0 else 0
    ax2.text(i, d + 0.005, f'{pct:.1f}% drop', ha='center', fontsize=9, color='black')

plt.suptitle('IoU & mAP Analysis — One-Step YOLOv8', fontsize=13)
plt.tight_layout()
plt.savefig('test_iou_map_analysis.png', dpi=150)
plt.show()

   IoU & mAP Results — One-Step YOLOv8
  Class               AP@50   AP@50-95
-------------------------------------------------------
  fall detected      0.5605     0.2801
  walk               0.4697     0.2749
  sit                0.4099     0.2259
-------------------------------------------------------
  mAP (mean)         0.4800     0.2603

  Interpretation:
  AP@50    = detection counted correct if IoU ≥ 0.50
  AP@50-95 = stricter — averaged over IoU 0.50 to 0.95
  Drop from AP@50 → AP@50-95 shows bounding box precision


<Figure size 1400x500 with 2 Axes>

---
## 5. Per-Class AP@50 Bar Chart

In [13]:
ap_values = metrics.box.ap50
colors    = ['#e74c3c', '#3498db', '#2ecc71']

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(CLASS_NAMES, ap_values, color=colors, edgecolor='black', width=0.5)
ax.bar_label(bars, fmt='%.4f', padding=4, fontsize=11, fontweight='bold')
ax.set_ylim(0, 1.05)
ax.set_ylabel('AP@50', fontsize=12)
ax.set_title('Per-Class AP@50 — One-Step YOLOv8 Test Set', fontsize=13)
ax.axhline(y=metrics.box.map50, color='black', linestyle='--',
           linewidth=1.2, label=f'mAP@50 = {metrics.box.map50:.4f}')
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('test_per_class_ap50.png', dpi=150)
plt.show()

<Figure size 700x400 with 1 Axes>

---
## 6. Confusion Matrix

In [14]:
# YOLO saves confusion_matrix.png automatically in the val results folder
val_dir = Path(metrics.save_dir)
cm_path = val_dir / 'confusion_matrix.png'

if cm_path.exists():
    img = cv2.cvtColor(cv2.imread(str(cm_path)), cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(7, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Confusion Matrix — Test Set', fontsize=13)
    plt.tight_layout()
    plt.savefig('test_confusion_matrix.png', dpi=150)
    plt.show()
    print('Saved to test_confusion_matrix.png')
else:
    print(f'Not found at {cm_path} — check val results directory')

<Figure size 700x600 with 1 Axes>

Saved to test_confusion_matrix.png


---
## 7. Condition Analysis — Normal vs Low-Light
Splits test images by filename prefix to compare model performance under different lighting.

In [15]:
# ── Separate test images by condition ────────────────────────────────────────
all_imgs = [f for f in TEST_IMG_DIR.iterdir()
            if f.suffix.lower() in ('.jpg', '.jpeg', '.png')]

normal_imgs   = [f for f in all_imgs
                 if not f.stem.lower().startswith(LOW_LIGHT_PREFIXES)]
lowlight_imgs = [f for f in all_imgs
                 if f.stem.lower().startswith(LOW_LIGHT_PREFIXES)]

print(f'Total test images  : {len(all_imgs)}')
print(f'Normal images      : {len(normal_imgs)}')
print(f'Low-light images   : {len(lowlight_imgs)}')

Total test images  : 183
Normal images      : 151
Low-light images   : 32


In [18]:
def predict_condition(img_list, label, conf=CONF_THRESHOLD, batch_size=10):
    """Run predictions in batches to avoid CPU timeout."""
    if not img_list:
        print(f'[{label}] No images found'); return Counter()

    counts = Counter()
    total_imgs = len(img_list)

    for i in range(0, total_imgs, batch_size):
        batch = img_list[i:i+batch_size]
        preds = model.predict(
            source  = [str(f) for f in batch],
            imgsz   = IMG_SIZE,
            conf    = conf,
            iou     = IOU_THRESHOLD,
            verbose = False
        )
        for r in preds:
            for cls in r.boxes.cls.tolist():
                counts[int(cls)] += 1

        print(f'  [{label}] {min(i+batch_size, total_imgs)}/{total_imgs} done...', end='\r')

    total = sum(counts.values())
    print(f'\n[{label}]  {total_imgs} images  →  {total} total detections')
    for cls_id, cnt in sorted(counts.items()):
        print(f'   {CLASS_NAMES[cls_id]:<16}: {cnt}')
    return counts

counts_normal   = predict_condition(normal_imgs,   'Normal')
counts_lowlight = predict_condition(lowlight_imgs, 'Low-Light')

  [Normal] 151/151 done...
[Normal]  151 images  →  275 total detections
   fall detected   : 46
   walk            : 165
   sit             : 64
  [Low-Light] 32/32 done...
[Low-Light]  32 images  →  75 total detections
   fall detected   : 14
   walk            : 40
   sit             : 21


In [19]:
# ── Side-by-side bar chart ────────────────────────────────────────────────────
norm_vals = [counts_normal.get(i, 0)   for i in range(len(CLASS_NAMES))]
low_vals  = [counts_lowlight.get(i, 0) for i in range(len(CLASS_NAMES))]

x = np.arange(len(CLASS_NAMES))
w = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
b1 = ax.bar(x - w/2, norm_vals, w, label='Normal',    color='#3498db', edgecolor='black')
b2 = ax.bar(x + w/2, low_vals,  w, label='Low-Light', color='#e67e22', edgecolor='black')
ax.bar_label(b1, padding=3, fontsize=10)
ax.bar_label(b2, padding=3, fontsize=10)
ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES, fontsize=11)
ax.set_ylabel('Number of Detections', fontsize=11)
ax.set_title('Detections by Class — Normal vs Low-Light Conditions', fontsize=13)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

# Annotate drop percentage
for i, (n, l) in enumerate(zip(norm_vals, low_vals)):
    if n > 0:
        drop = (n - l) / n * 100
        ax.text(i, max(n, l) + 5, f'-{drop:.0f}%',
                ha='center', fontsize=9, color='red', fontweight='bold')

plt.tight_layout()
plt.savefig('test_condition_comparison.png', dpi=150)
plt.show()

<Figure size 800x500 with 1 Axes>

---
## 8. Qualitative Results — Visualise Predictions

In [20]:
def visualise_predictions(img_list, n=6, conf=CONF_THRESHOLD, title='Predictions'):
    """Show a grid of n random images with predicted bounding boxes."""
    if not img_list:
        print('No images to show'); return

    sample = random.sample(img_list, min(n, len(img_list)))
    fig, axes = plt.subplots(2, 3, figsize=(16, 9))

    for ax, img_path in zip(axes.flatten(), sample):
        result  = model.predict(
            source  = str(img_path),
            imgsz   = IMG_SIZE,
            conf    = conf,
            iou     = IOU_THRESHOLD,
            verbose = False
        )[0]
        plotted = result.plot()   # BGR with boxes drawn by YOLO
        ax.imshow(cv2.cvtColor(plotted, cv2.COLOR_BGR2RGB))
        ax.set_title(img_path.stem, fontsize=8)
        ax.axis('off')

    # Hide unused subplots if fewer than 6 images
    for ax in axes.flatten()[len(sample):]:
        ax.axis('off')

    plt.suptitle(title, fontsize=13)
    plt.tight_layout()
    fname = title.replace(' ', '_').replace('—', '-') + '.png'
    plt.savefig(fname, dpi=150)
    plt.show()
    print(f'Saved: {fname}')

# Normal conditions
visualise_predictions(normal_imgs,   n=6, title='Test Predictions — Normal Conditions')

# Low-light conditions
visualise_predictions(lowlight_imgs, n=6, title='Test Predictions — Low-Light Conditions')

<Figure size 1600x900 with 6 Axes>

Saved: Test_Predictions_-_Normal_Conditions.png


<Figure size 1600x900 with 6 Axes>

Saved: Test_Predictions_-_Low-Light_Conditions.png


---
## 9. Viewpoint & Distance Analysis
Manually tag your test images by viewpoint and distance if you have sub-folders,
or use the filename prefix approach below.

In [21]:
# ── Define viewpoint groups by filename prefix ────────────────────────────────
# Update these prefixes to match YOUR filenames
VIEWPOINT_GROUPS = {
    'side_view'  : ('walk', 'fall_0', 'sit_0'),      # side angle images
    'top_down'   : ('top', 'above', 'overhead'),      # top-down angle
    'frontal'    : ('front', 'face'),                 # facing camera
}

DISTANCE_GROUPS = {
    'close'   : ('close', 'near'),
    'medium'  : ('mid', 'medium'),
    'far'     : ('far', 'distant'),
}

def group_by_prefix(img_list, groups):
    result = {k: [] for k in groups}
    result['other'] = []
    for f in img_list:
        matched = False
        for group_name, prefixes in groups.items():
            if f.stem.lower().startswith(prefixes):
                result[group_name].append(f)
                matched = True
                break
        if not matched:
            result['other'].append(f)
    return result

viewpoint_groups = group_by_prefix(all_imgs, VIEWPOINT_GROUPS)
print('Viewpoint groups:')
for g, imgs in viewpoint_groups.items():
    print(f'  {g}: {len(imgs)} images')

# ── Run prediction per viewpoint group ───────────────────────────────────────
vp_results = {}
for group_name, imgs in viewpoint_groups.items():
    if imgs:
        preds = model.predict(
            source=[str(f) for f in imgs],
            imgsz=IMG_SIZE, conf=CONF_THRESHOLD,
            iou=IOU_THRESHOLD, verbose=False
        )
        counts = Counter()
        for r in preds:
            for cls in r.boxes.cls.tolist():
                counts[int(cls)] += 1
        vp_results[group_name] = counts
        total = sum(counts.values())
        print(f'[{group_name}] {len(imgs)} images → {total} detections')

Viewpoint groups:
  side_view: 151 images
  top_down: 0 images
  frontal: 0 images
  other: 32 images
[side_view] 151 images → 276 detections
[other] 32 images → 75 detections


---
## 10. Single Image Test
Quick test on any image path you provide.

In [22]:
def test_single_image(img_path, conf=CONF_THRESHOLD):
    """Run model on a single image and display result."""
    img_path = Path(img_path)
    assert img_path.exists(), f'Image not found: {img_path}'

    result  = model.predict(
        source  = str(img_path),
        imgsz   = IMG_SIZE,
        conf    = conf,
        iou     = IOU_THRESHOLD,
        verbose = False
    )[0]

    # Print detections
    print(f'Image : {img_path.name}')
    print(f'Detections: {len(result.boxes)}')
    for box in result.boxes:
        cls  = int(box.cls[0])
        conf = float(box.conf[0])
        x1, y1, x2, y2 = [int(v) for v in box.xyxy[0].tolist()]
        print(f'  [{CLASS_NAMES[cls]}]  conf={conf:.3f}  box=({x1},{y1},{x2},{y2})')

    # Display
    plotted = result.plot()
    plt.figure(figsize=(8, 6))
    plt.imshow(cv2.cvtColor(plotted, cv2.COLOR_BGR2RGB))
    plt.title(img_path.name, fontsize=11)
    plt.axis('off')
    plt.tight_layout()
    plt.show()
    return result

# ── Example usage — change path to any image ─────────────────────────────────
# Picks a random test image automatically
random_img = random.choice(all_imgs)
test_single_image(random_img)

Image : sit_044.jpg
Detections: 1
  [walk]  conf=0.812  box=(0,59,56,208)


<Figure size 800x600 with 1 Axes>

ultralytics.engine.results.Results object with attributes:

boxes: ultralytics.engine.results.Boxes object
keypoints: None
masks: None
names: {0: 'fall detected', 1: 'walk', 2: 'sit'}
obb: None
orig_img: array([[[111, 133, 161],
        [111, 133, 161],
        [111, 133, 161],
        ...,
        [ 96, 107, 115],
        [ 81,  92, 100],
        [ 71,  82,  90]],

       [[111, 133, 161],
        [111, 133, 161],
        [111, 133, 161],
        ...,
        [ 93, 104, 112],
        [ 78,  89,  97],
        [ 68,  79,  87]],

       [[112, 134, 162],
        [112, 134, 162],
        [112, 134, 162],
        ...,
        [ 90, 100, 110],
        [ 74,  84,  94],
        [ 64,  74,  84]],

       ...,

       [[142, 164, 192],
        [145, 167, 195],
        [149, 171, 199],
        ...,
        [147, 142, 141],
        [159, 153, 148],
        [145, 135, 128]],

       [[144, 165, 192],
        [145, 166, 193],
        [145, 167, 195],
        ...,
        [140, 135, 136],
        [1

---
## 12. Summary Table

In [23]:
import warnings
warnings.filterwarnings('ignore')

# Build summary dataframe
summary = pd.DataFrame({
    'Class'      : CLASS_NAMES,
    'AP@50'      : [round(v, 4) for v in metrics.box.ap50],
    'Normal Det' : [counts_normal.get(i, 0)   for i in range(3)],
    'LowLight Det': [counts_lowlight.get(i, 0) for i in range(3)],
})
summary['Drop (%)'] = (
    (summary['Normal Det'] - summary['LowLight Det'])
    / summary['Normal Det'].replace(0, 1) * 100
).round(1)

print('='*60)
print('   ONE-STEP YOLO TEST RESULTS SUMMARY')
print('='*60)
print(f'  Overall mAP@50    : {metrics.box.map50:.4f}')
print(f'  Overall mAP@50-95 : {metrics.box.map:.4f}')
print(f'  Overall Precision : {metrics.box.mp:.4f}')
print(f'  Overall Recall    : {metrics.box.mr:.4f}')
print(f'  Test images       : {len(all_imgs)}')
print(f'  Normal images     : {len(normal_imgs)}')
print(f'  Low-light images  : {len(lowlight_imgs)}')
print('='*60)
print(summary.to_string(index=False))
print('='*60)

   ONE-STEP YOLO TEST RESULTS SUMMARY
  Overall mAP@50    : 0.4800
  Overall mAP@50-95 : 0.2603
  Overall Precision : 0.4425
  Overall Recall    : 0.5979
  Test images       : 183
  Normal images     : 151
  Low-light images  : 32
        Class  AP@50  Normal Det  LowLight Det  Drop (%)
fall detected 0.5605          46            14      69.6
         walk 0.4697         165            40      75.8
          sit 0.4099          64            21      67.2
